# Results analysis (all folds)

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent                   

RUNS = {                                   # variants of Table 4.2
    "V1":  "runs only masked loss(augme and no)/mit_noaug",
    "V2":  "runs only masked loss(augme and no)/mit_aug",
    "V3":  "runsola/noaug_w0",
    "V4":  "runsola/aug_w0",
    "V5":  "runsola/noaug_w03",
    "V6":  "runsola/aug_w03",
    "V7":  "runsola/noaug_w06",
    "V8":  "runsola/aug_w06",
    "V9":  "noaug_w0_lr1e3",
    "V10": "noaug_w0_lr1e3_d64L2",
    "V11": "noaug_w0_lr1e3_do05",
    "V12": "noaug_w0_lr7e-4",
}
MAIN = "V9"                                # main variant

BARNETT = {                                # Barnett & Onnela on the same gaps
    "wp":     "noaug_w0_lr1e3/runs_barnett_wp",
    "nowp":   "noaug_w0_lr1e3/runs_barnett",
    "static": "noaug_w0_lr1e3/runs_barnett_nonmoving",
}

missing = [k for k, v in RUNS.items() if not (ROOT / v).exists()]
RUNS = {k: v for k, v in RUNS.items() if k not in missing}
if missing:
    print("not found (omitted from Tables 4.2 and 4.8):", missing)

N_BOOT, SEED = 1000, 0                     # bootstrap over gaps
FAM_BINS = [-float("inf"), 1, 3, 10, float("inf")]   
FAM_LABELS = ["0", "1-2", "3-9", "10+"]
BAD_DAYS = ["2025-07-05", "2025-11-30"]    # the two days of Section 4.6

## Loading

Only the masked points are kept. For each gap the sum of squared errors etc. is computed

In [ ]:
import glob, json, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pyproj import Transformer

pd.set_option("display.width", 200)
to_egsa = Transformer.from_crs("epsg:4326", "epsg:2100", always_xy=True)
COLS = ["fold", "day", "win_id", "win_num", "gap_len", "moving", "is_hidden", "err_m", "path_err_m",
        "fam_days", "err_norm", "step_cv_true", "step_cv_pred"]


def pick(run, pattern):
    d = ROOT / run
    f = sorted(d.glob(f"summary/{pattern}"))
    return f if f else sorted(d.glob(f"fold*/{pattern}"))


def gap_key(d):
    w = d.win_id.astype(str)
    return np.where(w.str.startswith("f"), w, "f" + d.fold.astype(str) + "_" + w)


def load(run, scen=None, pattern=None):
    files = pick(run, pattern or f"pred_fold*_{scen}.csv")
    assert len(files) == 4, f"{run}: found {len(files)} files instead of 4"
    parts = []
    for f in files:
        d = pd.read_csv(f, usecols=COLS)
        d = d[d.is_hidden == 1].copy()
        d["gap"] = gap_key(d)
        parts.append(d)
    return pd.concat(parts, ignore_index=True)


def per_gap(h):
    return h.assign(se=h.err_m ** 2).groupby("gap", sort=False).agg(
        se=("se", "sum"), n=("se", "size"), fold=("fold", "first"), day=("day", "first"),
        win=("win_num", "first"), L=("gap_len", "first"), moving=("moving", "first"),
        fam=("fam_days", "first"), cv_true=("step_cv_true", "first"), cv_pred=("step_cv_pred", "first"))


P = {s: load(RUNS[MAIN], s) for s in ["wp", "nowp"]}
G = {s: per_gap(h) for s, h in P.items()}
assert G["wp"].index.is_unique and len(G["wp"]) == len(G["nowp"]), "duplicate or different gaps"
print({s: f"{len(g):,} gaps, {len(P[s]):,} masked points" for s, g in G.items()})

## Helper functions

-  RMSE with a 95% confidence interval, resampling whole gaps.
-  comparison of two methods on the same gaps (ΔRMSE, mean difference per gap, share of gaps where the second one improves).

In [ ]:
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))


def rmse_ci(g, n_boot=N_BOOT, seed=SEED):
    """RMSE and 95% CI with bootstrap over gaps."""
    se, n = g.se.to_numpy(), g.n.to_numpy()
    rng = np.random.default_rng(seed)
    b = [np.sqrt(se[i].sum() / n[i].sum()) for i in (rng.integers(0, len(se), len(se)) for _ in range(n_boot))]
    return np.sqrt(se.sum() / n.sum()), np.percentile(b, 2.5), np.percentile(b, 97.5)


def paired(ga, gb, n_boot=N_BOOT, seed=SEED):
    """ ΔRMSE and mean RMSE difference per gap, with 95% CI (same gaps in every resample)."""
    m = ga[["se", "n"]].join(gb[["se", "n"]], lsuffix="_a", rsuffix="_b", how="inner")
    assert len(m) == len(ga) == len(gb), "the two methods do not have the same gaps"
    sa, na, sb, nb = (m[c].to_numpy() for c in ["se_a", "n_a", "se_b", "n_b"])
    d = np.sqrt(sb / nb) - np.sqrt(sa / na)                  # RMSE difference per gap
    rng = np.random.default_rng(seed)
    bd, bm = [], []
    for _ in range(n_boot):
        i = rng.integers(0, len(m), len(m))
        bd.append(np.sqrt(sb[i].sum() / nb[i].sum()) - np.sqrt(sa[i].sum() / na[i].sum()))
        bm.append(d[i].mean())
    q = lambda x: np.percentile(x, [2.5, 97.5])
    return dict(d_rmse=np.sqrt(sb.sum() / nb.sum()) - np.sqrt(sa.sum() / na.sum()), d_rmse_ci=q(bd),
                mean_diff=d.mean(), mean_diff_ci=q(bm), b_better=(d < 0).mean())


def fam_bin(f):
    """Familiarity category."""
    return pd.cut(f, FAM_BINS, labels=FAM_LABELS, right=False)


def fmt_ci(v, lo, hi):
    return f"{v:,.0f} [{lo:,.0f}–{hi:,.0f}]"


def err_row(h, g):
    """RMSE [CI], median and p90 of the position error."""
    return {"Gaps": len(g), "RMSE": fmt_ci(*rmse_ci(g)),
            "Median": round(h.err_m.median()), "p90": round(h.err_m.quantile(0.9))}


def by_L(h, g, fn):
    rows = {L: fn(h[h.gap_len == L], g[g.L == L]) for L in sorted(g.L.unique())}
    rows["All"] = fn(h, g)
    return pd.DataFrame(rows).T

## Table 4.1: composition of the test set

In [ ]:
g = G["wp"]
t41 = g.groupby("fold").agg(Days=("day", "nunique"), Windows=("win", "nunique"),
                            No_movement=("moving", lambda m: int((m == 0).sum())),
                            With_movement=("moving", "sum"))
mv = g[g.moving == 1]
t41["Familiar"] = mv.groupby("fold").fam.apply(lambda f: int((f >= 1).sum()))
t41["Unfamiliar"] = mv.groupby("fold").fam.apply(lambda f: int((f < 1).sum()))
t41.loc["Total"] = t41.sum()
print(t41.to_string())
print(f"\ngaps: {len(g):,} | per L: {g.groupby('L').size().to_dict()}")
print(f"gaps with movement: {len(mv):,} ({len(mv) / len(g):.0%}) from {mv.groupby(['fold', 'win']).ngroups} windows")

## Table 4.2: variants on validation

RMSE of the familiar gaps with movement on validation, at epoch E of each fold, averaged over the 4 folds.

In [ ]:
rows = {}
for name, run in RUNS.items():
    Es, v = [], []
    for k in range(1, 5):
        E = json.load(open(pick(run, f"E_fold{k}.json")[0]))["E"]
        c = pd.read_csv(pick(run, f"val_curve_fold{k}.csv")[0])
        Es.append(E); v.append(float(c.loc[c.n_epochs == E, "rmse_mov1_fam"].iloc[0]))
    rows[name] = {"E per fold": Es, "RMSE val (m)": round(np.mean(v), 1)}
t42 = pd.DataFrame(rows).T
print(t42.to_string())

## Tables 4.3–4.5: gaps with movement per gap length (main variant)

In [ ]:
H = {s: P[s][P[s].moving == 1] for s in P}
GM = {s: G[s][G[s].moving == 1] for s in G}

t43 = pd.concat({s: by_L(H[s], GM[s], err_row) for s in ["wp", "nowp"]}, axis=1)
print("Table 4.3: position error (m)\n" + t43.to_string())


def path_row(h, g):
    return {"path RMSE": round(rmse(h.path_err_m)), "path median": round(h.path_err_m.median()),
            "norm median": round(h.err_norm.median(), 2), "norm p90": round(h.err_norm.quantile(0.9), 2)}


t44 = pd.concat({s: by_L(H[s], GM[s], path_row) for s in ["wp", "nowp"]}, axis=1)
print("\nTable 4.4: path error (m) and normalized error\n" + t44.to_string())

cv = lambda s: by_L(H[s], GM[s], lambda h, g: {"true": g.cv_true.median(), "pred": g.cv_pred.median()})
t45 = pd.DataFrame({"True": cv("wp")["true"], "With waypoints": cv("wp")["pred"],
                    "Without waypoints": cv("nowp")["pred"]}).astype(float).round(2)
print("\nTable 4.5: coefficient of variation of steps (median over gaps)\n" + t45.to_string())

## Gaps without movement (main variant)

In [ ]:
hs, gs = P["wp"][P["wp"].moving == 0], G["wp"][G["wp"].moving == 0]
unf = gs.fam < 1
r, lo, hi = rmse_ci(gs)
print(f"{len(gs):,} gaps | RMSE {fmt_ci(r, lo, hi)} | median {hs.err_m.median():.0f} | p90 {hs.err_m.quantile(.9):.0f} m")
print(f"unfamiliar: {int(unf.sum())} gaps, RMSE {np.sqrt(gs[unf].se.sum() / gs[unf].n.sum()):,.0f} m, "
      f"{gs[unf].se.sum() / gs.se.sum():.0%} of the squared error | without them: "
      f"{np.sqrt(gs[~unf].se.sum() / gs[~unf].n.sum()):.0f} m")
f10 = gs.fam >= 10
print(f"familiarity ≥ 10: {int(f10.sum()):,} gaps, RMSE {np.sqrt(gs[f10].se.sum() / gs[f10].n.sum()):.0f} m")
hn = P["nowp"][P["nowp"].moving == 0]
print(f"without waypoints: RMSE {rmse(hn.err_m):.0f} | median {hn.err_m.median():.0f} | p90 {hn.err_m.quantile(.9):.0f} m "
      f"({len(hn) - len(hs)} more masked points, from the few gaps with a waypoint)")

## Tables 4.6–4.7: per fold and per familiarity (gaps with movement)

In [ ]:
t46 = H["wp"].groupby("fold").err_m.agg(RMSE=rmse, Median="median").round(0)
t46.insert(0, "Gaps", GM["wp"].groupby("fold").size())
t46.loc["All"] = [len(GM["wp"]), round(rmse(H["wp"].err_m)), round(H["wp"].err_m.median())]
print("Table 4.6\n" + t46.to_string())


def fam_table(h, g):
    h, g = h.assign(fb=fam_bin(h.fam_days)), g.assign(fb=fam_bin(g.fam))
    return pd.DataFrame({b: {"Gaps": int((g.fb == b).sum()), "RMSE": fmt_ci(*rmse_ci(g[g.fb == b])),
                             "Median": round(h[h.fb == b].err_m.median())} for b in FAM_LABELS}).T


t47 = pd.concat({s: fam_table(H[s], GM[s]) for s in ["wp", "nowp"]}, axis=1)
print("\nTable 4.7\n" + t47.to_string())

## Table 4.8: pairwise comparison of variants

Gaps with movement, with waypoints. ΔRMSE = RMSE_B − RMSE_A and mean difference per gap, with 95% CI. 

In [ ]:
PAIRS = [("ORT", "V1", "V3"), ("ORT", "V2", "V4"),
         ("step loss 0,3", "V3", "V5"), ("step loss 0,3", "V4", "V6"),
         ("step loss 0,6", "V3", "V7"), ("step loss 0,6", "V4", "V8"),
         ("Augmentation", "V1", "V2"), ("Augmentation", "V3", "V4"), ("Augmentation", "V5", "V6"), ("Augmentation", "V7", "V8"),
         ("Learning rate", "V3", "V9"),
         ("Model size", "V9", "V10"), ("Dropout 0,5", "V9", "V11"), ("Learning rate 7e-4", "V9", "V12")]

HV, GV = {}, {}                              # gaps with movement of each variant (wp scenario)
for name, run in RUNS.items():
    h = H["wp"] if name == MAIN else load(run, "wp").query("moving == 1")
    HV[name], GV[name] = h, per_gap(h)

summary = pd.DataFrame({n: {"RMSE": rmse(h.err_m), "Median": h.err_m.median(),
                            "path median": h.path_err_m.median(), "CV prediction": GV[n].cv_pred.median(),
                            "RMSE unfamiliar": rmse(h[h.fam_days < 1].err_m)}
                        for n, h in HV.items()}).T.astype(float).round(2)
print("Per variant (test, gaps with movement, wp)\n" + summary.to_string())

rows = []
for opt, a, b in PAIRS:
    if a not in GV or b not in GV:
        continue
    p = paired(GV[a], GV[b])
    rows.append({"Option": opt, "Comparison": f"{a} → {b}",
                 "ΔRMSE": f"{p['d_rmse']:.0f} [{p['d_rmse_ci'][0]:.0f}, {p['d_rmse_ci'][1]:.0f}]",
                 "Mean difference": f"{p['mean_diff']:.0f} [{p['mean_diff_ci'][0]:.0f}, {p['mean_diff_ci'][1]:.0f}]",
                 "Impr. (%)": round(100 * p["b_better"])})
t48 = pd.DataFrame(rows)
print("\nTable 4.8\n" + t48.to_string(index=False))

## Error analysis (Section 4.6)


In [ ]:
g = GM["wp"].sort_values("se", ascending=False)
top = g.head(10)
print(f"10 worst gaps: {top.se.sum() / g.se.sum():.0%} of the squared error | "
      f"windows {top.groupby(['fold', 'win']).ngroups} | days {sorted(top.day.unique())}")
bad = g.day.isin(BAD_DAYS)
print(f"gaps of the two days: {int(bad.sum())}, {g[bad].se.sum() / g.se.sum():.0%} of the squared error")
h = H["wp"][~H["wp"].day.isin(BAD_DAYS)]
print(f"without the two days: RMSE {rmse(h.err_m):.0f} m, median {h.err_m.median():.0f} m | per fold: "
      f"{h.groupby('fold').err_m.apply(rmse).round(0).astype(int).tolist()}")

# gaps of Figure 4.3: interval from the first to the last masked point
FIG_GAPS = {2: ["w285_L31", "w286_L31"], 4: ["w1114_L31", "w1116_L31"]}
for F, ws in FIG_GAPS.items():
    d = pd.read_csv(pick(RUNS[MAIN], f"pred_fold{F}_wp.csv")[0], usecols=["win_id", "real_time", "is_hidden"])
    for w in ws:
        t = pd.to_datetime(d[(d.win_id.str.endswith(w)) & (d.is_hidden == 1)].real_time)
        print(f"  fold {F} {w}: {t.min():%d/%m %H:%M}–{t.max():%H:%M}")

## Figure 4.4: training curves (main variant)

(a) Train error = 100 × √L_MIT. (b) Selection criterion for epoch E (3-epoch moving average), in meters: `crit_ma3` × minimum RMSE of the familiar gaps with movement. 

In [ ]:
VAL_CURVE = "crit"                          
CURRICULUM_EPOCHS = [5, 10, 20, 30, 40]
plt.rcParams.update({"font.family": "serif"})
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for k, c in zip(range(1, 5), ["#0072B2", "#D55E00", "#D62728", "#7B3294"]):
    v = pd.read_csv(pick(RUNS[MAIN], f"val_curve_fold{k}.csv")[0])
    E = json.load(open(pick(RUNS[MAIN], f"E_fold{k}.json")[0]))["E"]
    tr = 100 * np.sqrt(v.coord_loss)
    va = (v.crit_ma3 * v.rmse_mov1_fam.min() if VAL_CURVE == "crit"
          else v.rmse_mov1_fam.rolling(3, center=True).mean())
    for a, y in zip(ax, [tr, va]):
        a.plot(v.n_epochs, y, color=c, lw=1.8, label=f"Fold {k}")
        a.plot(E, y[v.n_epochs == E].iloc[0], "o", color=c, ms=8, mec="white", zorder=3)
for a, t, yl in zip(ax, ["(a) Training", "(b) Validation"], ["Training error (m)", "Validation error (m)"]):
    for e in CURRICULUM_EPOCHS:
        a.axvline(e, ls="--", lw=1, color="#999999")
    a.plot([], [], "o", color="#333333", label="Selected epoch\n(epoch > 40)")
    a.set_yscale("log"); a.set_xlim(0, 100); a.set_ylim(100, 3500)
    a.set_yticks([100, 200, 300, 500, 1000, 2000, 3000], ["100", "200", "300", "500", "1000", "2000", "3000"])
    a.minorticks_off(); a.grid(axis="y", alpha=0.3)
    a.spines[["top", "right"]].set_visible(False)
    a.set_title(t, fontsize=13); a.set_xlabel("Epoch"); a.set_ylabel(yl); a.legend(loc="upper right")
fig.tight_layout(); plt.show()
print("E per fold:", [json.load(open(pick(RUNS[MAIN], f"E_fold{k}.json")[0]))["E"] for k in range(1, 5)])

## Tables 4.9–4.12: comparison with Barnett & Onnela

The method has been evaluated on the same gaps. ΔRMSE and the mean difference are given as B&O − V9.

In [ ]:
B = {s: load(run, pattern="MC100_BSTAR_pred_fold*.csv") for s, run in BARNETT.items()}
BG = {s: per_gap(h) for s, h in B.items()}
for s in ["wp", "nowp"]:
    assert set(BG[s].index) == set(GM[s].index), f"B&O {s}: different gaps from the model"
assert set(BG["static"].index) == set(gs.index), "B&O static: different gaps from the model"

t49 = pd.concat({"B&O": by_L(B["wp"], BG["wp"], err_row), MAIN: t43["wp"]}, axis=1)
print("Table 4.9 (with waypoints)\n" + t49.to_string())

for s in ["wp", "nowp"]:
    p = paired(GM[s], BG[s])
    print(f"\n{s}: B&O RMSE {fmt_ci(*rmse_ci(BG[s]))}, median {B[s].err_m.median():.0f} | "
          f"{MAIN} RMSE {fmt_ci(*rmse_ci(GM[s]))}, median {H[s].err_m.median():.0f} | "
          f"ΔRMSE {p['d_rmse']:.0f} {np.round(p['d_rmse_ci']).tolist()} | "
          f"{MAIN} better on {1 - p['b_better']:.0%} of the gaps")
print(f"nowp, L = 31: median {MAIN} {H['nowp'][H['nowp'].gap_len == 31].err_m.median():.0f} | "
      f"B&O {B['nowp'][B['nowp'].gap_len == 31].err_m.median():.0f}")

In [ ]:
# concentration of the error, the two days and spread per fold
for name, h, g in [("B&O", B["wp"], BG["wp"]), (MAIN, H["wp"], GM["wp"])]:
    d = h.day.isin(BAD_DAYS)
    print(f"{name}: 10 worst gaps {g.se.nlargest(10).sum() / g.se.sum():.0%} | two days RMSE {rmse(h[d].err_m):,.0f} | "
          f"without them {rmse(h[~d].err_m):.0f} | std per fold {h.groupby('fold').err_m.apply(rmse).std():.0f} m")

t410 = pd.DataFrame({n: {"path RMSE": round(rmse(h.path_err_m)), "path median": round(h.path_err_m.median()),
                         "norm median": round(h.err_norm.median(), 2), "norm p90": round(h.err_norm.quantile(.9), 2),
                         "CV": round(g.cv_pred.median(), 2)}
                     for n, h, g in [("B&O", B["wp"], BG["wp"]), (MAIN, H["wp"], GM["wp"])]}).T
print(f"\nTable 4.10 (true CV {GM['wp'].cv_true.median():.2f})\n" + t410.to_string())

In [ ]:
# Figure 4.5 (fold 1, w255_L20): RMSE per segment between waypoints
FIG_GAP, FIG_FOLD = "w255_L20", 1
cfg = json.load(open(pick(RUNS[MAIN], "norm_config.json")[0]))
raw = pd.read_csv(ROOT / BARNETT["wp"] / f"fold{FIG_FOLD}" / f"MC100_pred_fold{FIG_FOLD}.csv")
for name, d in [("B&O", pd.read_csv(ROOT / BARNETT["wp"] / f"fold{FIG_FOLD}" / f"MC100_BSTAR_pred_fold{FIG_FOLD}.csv")),
                (MAIN, pd.read_csv(pick(RUNS[MAIN], f"pred_fold{FIG_FOLD}_wp.csv")[0])), ("B&O before bridging", raw)]:
    g = d[d.win_id.str.endswith(FIG_GAP)].sort_values("step_index")
    part = np.cumsum(g.point_type.isin(["Anchor", "Waypoint"]).to_numpy())    # 1 = up to the 1st waypoint
    h = (g.is_hidden == 1).to_numpy()
    e1, e2 = g.err_m.to_numpy()[h & (part == 1)], g.err_m.to_numpy()[h & (part > 1)]
    line = f"{name}: RMSE up to the 1st waypoint {rmse(e1):.0f} m, after {rmse(e2):.0f} m"
    if d is raw:
        xs, ys = to_egsa.transform(g.lon.to_numpy()[h & (part == 1)], g.lat.to_numpy()[h & (part == 1)])
        line += f" | median distance from home up to the 1st waypoint {np.median(np.hypot(xs - cfg['x_home'], ys - cfg['y_home'])):.0f} m"
    print(line)

In [ ]:
# Table 4.11: per familiarity
fb = fam_bin(GM["wp"].fam)
rows = {}
for b in FAM_LABELS:
    idx = GM["wp"].index[fb == b]
    gb, gm = BG["wp"].loc[idx], GM["wp"].loc[idx]
    hb, hm = B["wp"][B["wp"].gap.isin(idx)], H["wp"][H["wp"].gap.isin(idx)]
    rows[b] = {"Gaps": len(idx), "B&O RMSE": fmt_ci(*rmse_ci(gb)), "B&O median": round(hb.err_m.median()),
               f"{MAIN} RMSE": fmt_ci(*rmse_ci(gm)), f"{MAIN} median": round(hm.err_m.median()),
               "B&O better (%)": round(100 * paired(gm, gb)["b_better"])}
print("Table 4.11\n" + pd.DataFrame(rows).T.to_string())

In [ ]:
# Table 4.12: gaps without movement
rows = {}
for name, h, g in [("B&O", B["static"], BG["static"]), (MAIN, hs, gs)]:
    f = g.fam.reindex(g.index)
    rows[name] = {"RMSE": fmt_ci(*rmse_ci(g)), "Median": round(h.err_m.median()), "p90": round(h.err_m.quantile(.9)),
                  f"RMSE fam. 0 ({int((f < 1).sum())})": round(np.sqrt(g[f < 1].se.sum() / g[f < 1].n.sum())),
                  f"RMSE fam. ≥ 10 ({int((f >= 10).sum()):,})": round(np.sqrt(g[f >= 10].se.sum() / g[f >= 10].n.sum()))}
print("Table 4.12\n" + pd.DataFrame(rows).T.to_string())
print(f"B&O better on {paired(gs, BG['static'])['b_better']:.0%} of the gaps without movement")